
# Employee Turnover Analytics - Advanced End-to-End ML Project

**Author:** Abdul Samad Shaikh  
**Dataset:** HR_comma_sep (Employee Turnover Dataset)

This notebook covers:

- Data Quality Assessment
- Advanced EDA
- Correlation Analysis
- KMeans Employee Segmentation
- SMOTE Class Balancing
- Logistic Regression
- Random Forest
- Gradient Boosting
- 5-Fold Cross Validation
- ROC/AUC Comparison
- Confusion Matrix Analysis
- Feature Importance
- Employee Risk Segmentation
- Retention Strategy Recommendations


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.cluster import KMeans

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(10,6)


In [ ]:

df = pd.read_excel('HR_comma_sep.xlsx')

print('Shape:', df.shape)
df.head()


## 1. Data Quality Check

In [ ]:

print(df.info())

print('\nMissing Values')
print(df.isnull().sum())

print('\nDuplicate Records:', df.duplicated().sum())


## 2. Exploratory Data Analysis

In [ ]:

num_cols = df.select_dtypes(include=np.number).columns

plt.figure(figsize=(10,8))
sns.heatmap(df[num_cols].corr(),
            annot=True,
            cmap='coolwarm',
            fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


In [ ]:

fig, ax = plt.subplots(1,3, figsize=(18,5))

sns.histplot(df['satisfaction_level'], kde=True, ax=ax[0])
ax[0].set_title('Employee Satisfaction')

sns.histplot(df['last_evaluation'], kde=True, ax=ax[1])
ax[1].set_title('Employee Evaluation')

sns.histplot(df['average_montly_hours'], kde=True, ax=ax[2])
ax[2].set_title('Average Monthly Hours')

plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(10,5))
sns.countplot(data=df,
              x='number_project',
              hue='left')

plt.title('Project Count vs Employee Turnover')
plt.show()



### Inference

- Employees with very low project involvement show higher turnover.
- Employees with excessive project load also tend to leave.
- Moderate workload is associated with better retention.


## 3. Employee Clustering

In [ ]:

left_emp = df[df['left']==1][
    ['satisfaction_level',
     'last_evaluation']
]

kmeans = KMeans(n_clusters=3,
                random_state=42)

left_emp['cluster'] = kmeans.fit_predict(left_emp)

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=left_emp,
    x='satisfaction_level',
    y='last_evaluation',
    hue='cluster',
    palette='Set1'
)

plt.title('Employees Who Left - KMeans Clusters')
plt.show()

left_emp.groupby('cluster').mean()



### Cluster Interpretation

1. Highly dissatisfied employees.
2. High performers experiencing burnout.
3. Moderate performers with declining engagement.


## 4. Preprocessing + SMOTE

In [ ]:

X = df.drop('left', axis=1)
y = df['left']

cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

X_cat = pd.get_dummies(X[cat_cols], drop_first=True)

X_final = pd.concat(
    [X[num_cols], X_cat],
    axis=1
)

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

smote = SMOTE(random_state=123)

X_train_sm, y_train_sm = smote.fit_resample(
    X_train,
    y_train
)

print('Original Train Shape:', X_train.shape)
print('After SMOTE:', X_train_sm.shape)


## 5. Model Training + 5 Fold Cross Validation

In [ ]:

models = {
    'Logistic Regression':
        LogisticRegression(max_iter=2000),
    'Random Forest':
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),
    'Gradient Boosting':
        GradientBoostingClassifier(
            random_state=42
        )
}

results = {}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, model in models.items():
    
    preds = cross_val_predict(
        model,
        X_train_sm,
        y_train_sm,
        cv=cv
    )
    
    print('='*70)
    print(name)
    print(classification_report(
        y_train_sm,
        preds
    ))


## 6. ROC Curve and Model Evaluation

In [ ]:

plt.figure(figsize=(10,7))

for name, model in models.items():
    
    model.fit(X_train_sm, y_train_sm)
    
    prob = model.predict_proba(X_test)[:,1]
    
    auc = roc_auc_score(y_test, prob)
    
    fpr, tpr, _ = roc_curve(y_test, prob)
    
    plt.plot(fpr, tpr,
             label=f'{name} AUC={auc:.3f}')
    
    results[name] = auc

plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title('ROC Curve Comparison')
plt.show()

results


In [ ]:

for name, model in models.items():
    
    model.fit(X_train_sm, y_train_sm)
    
    pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, pred)
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm,
                annot=True,
                fmt='d',
                cmap='Blues')
    
    plt.title(name)
    plt.show()



### Why Recall Matters

For employee turnover prediction, missing a resigning employee is more costly than incorrectly flagging a loyal employee.

Therefore:

**Recall is the primary metric**, supported by ROC-AUC and F1 Score.


## 7. Retention Strategy Engine

In [ ]:

best_model_name = max(results,
                      key=results.get)

best_model = models[best_model_name]

best_model.fit(X_train_sm, y_train_sm)

turnover_prob = best_model.predict_proba(X_test)[:,1]

risk = pd.DataFrame({
    'Turnover_Probability': turnover_prob
})

def zone(x):
    
    if x < 0.20:
        return 'Safe Zone'
    elif x < 0.60:
        return 'Low Risk'
    elif x < 0.90:
        return 'Medium Risk'
    else:
        return 'High Risk'

risk['Zone'] = risk[
    'Turnover_Probability'
].apply(zone)

risk.head()


In [ ]:

risk['Zone'].value_counts().plot(
    kind='bar'
)

plt.title('Employee Risk Segmentation')
plt.show()

risk['Zone'].value_counts()



# Retention Strategies

### Safe Zone (Green)
- Continue recognition programs
- Maintain engagement

### Low Risk Zone (Yellow)
- Career discussions
- Skill development plans

### Medium Risk Zone (Orange)
- Manager intervention
- Compensation review
- Workload balancing

### High Risk Zone (Red)
- Immediate HR engagement
- Personalized retention package
- Promotion or role redesign review
